# Compute Engine 인스턴스 생성 및 Ops Agent 정책 설정

이 노트북은 GCP CLI(`gcloud`)를 사용하여 Compute Engine 인스턴스를 생성하고, Google Cloud Ops Agent 정책을 등록합니다.

### 사전 확인 사항
- `gcloud auth login`을 통해 로그인된 상태여야 합니다.
- 현재 프로젝트: `sesac-dev-400904`

## [비용 분석] 동일 사양(e2-medium + 10GB pd-balanced) 리전별 요금 조회 및 최저가 Top 3

현재 생성하려는 VM의 사양:
- **머신 유형**: `e2-medium` (2 vCPU / 지속 1 vCPU, 4GB RAM)
- **부팅 디스크**: `pd-balanced` 10GB
- **네트워크/프로비저닝**: `STANDARD` (On-Demand), `PREMIUM` Network

아래 셀을 실행하면 Google Cloud Billing Catalog API를 실시간 호출하여 전 세계 모든 GCP 리전의 단가를 비교하고 **가장 저렴한 리전 Top 3**를 출력합니다.

In [ ]:
import urllib.request
import json
import subprocess
import ssl

# 1. GCP 인증 토큰 획득
ctx = ssl._create_unverified_context()
token = subprocess.check_output(["gcloud", "auth", "print-access-token"]).decode().strip()
url = "https://cloudbilling.googleapis.com/v1/services/6F81-5844-456A/skus?pageSize=5000"
headers = {"Authorization": f"Bearer {token}"}

region_core, region_ram, region_disk = {}, {}, {}
page_token = ""

print("⏳ Google Cloud Billing Catalog API에서 리전별 단가를 조회 중입니다...")

while True:
    req_url = url + ("&pageToken=" + page_token if page_token else "")
    req = urllib.request.Request(req_url, headers=headers)
    with urllib.request.urlopen(req, context=ctx) as resp:
        data = json.load(resp)
    for s in data.get("skus", []):
        desc = s.get("description", "")
        regions = s.get("serviceRegions", [])
        if not regions:
            continue
        r = regions[0]
        rates = s.get("pricingInfo", [{}])[0].get("pricingExpression", {}).get("tieredRates", [{}])
        if not rates:
            continue
        p_obj = rates[0].get("unitPrice", {})
        price = float(p_obj.get("units", 0)) + float(p_obj.get("nanos", 0)) / 1e9

        if desc.startswith("E2 Instance Core running in"):
            region_core[r] = price
        elif desc.startswith("E2 Instance Ram running in"):
            region_ram[r] = price
        elif "Balanced PD Capacity" in desc and "Regional" not in desc and "Snapshot" not in desc:
            region_disk[r] = price

    page_token = data.get("nextPageToken")
    if not page_token:
        break

# 2. e2-medium (1 Core + 4GB RAM, 730시간) + 10GB pd-balanced 비용 계산
results = []
for r in set(region_core.keys()) & set(region_ram.keys()):
    core_p = region_core[r]
    ram_p = region_ram[r]
    disk_p = region_disk.get(r, 0.10)
    hourly_vm = (1 * core_p) + (4 * ram_p)
    monthly_vm = hourly_vm * 730
    monthly_disk = disk_p * 10
    total_monthly = monthly_vm + monthly_disk
    results.append({
        "Region": r,
        "Hourly Total ($)": round(hourly_vm + (monthly_disk / 730), 4),
        "VM Monthly ($)": round(monthly_vm, 2),
        "Disk Monthly ($)": round(monthly_disk, 2),
        "Total Monthly ($)": round(total_monthly, 2)
    })

results.sort(key=lambda x: x["Total Monthly ($)"])

# 3. 결과 출력
print("=" * 75)
print("🏆 [e2-medium + 10GB pd-balanced 최저가 리전 Top 3]")
print("=" * 75)
for idx, res in enumerate(results[:3], 1):
    print(f"{idx}위: {res['Region']}")
    print(f"   - 총 예상 비용: ${res['Total Monthly ($)']:.2f} / 월 (시간당 ${res['Hourly Total ($)']:.4f})")
    print(f"   - 세부: VM ${res['VM Monthly ($)']:.2f}/월, 10GB 디스크 ${res['Disk Monthly ($)']:.2f}/월\n")

# pandas DataFrame으로 표 출력 (설치되어 있는 경우)
try:
    import pandas as pd
    df = pd.DataFrame(results[:10])
    df.index = df.index + 1
    print("📊 [상위 10개 리전 비교표]")
    display(df)
except Exception:
    pass


## 1. 인스턴스 생성 및 Ops Agent 정책 일괄 실행
아래 셀을 실행하면 다음 작업이 순차적으로 진행됩니다:
1. `instance-20260914-054744` VM 인스턴스 생성 (e2-medium, debian-13, my-vpc)
2. Ops Agent 정책 설정을 위한 `config.yaml` 파일 생성
3. Ops Agent 정책(`goog-ops-agent-v2-template-1-7-0-us-central1-c`) 등록

In [ ]:
%%bash
gcloud compute instances create instance-20260915-143200 \
    --project=sesac-dev-400904 \
    --zone=us-central1-c \
    --machine-type=e2-medium \
    --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet=my-vpc \
    --metadata=enable-osconfig=TRUE \
    --maintenance-policy=MIGRATE \
    --provisioning-model=STANDARD \
    --service-account=902882112756-compute@developer.gserviceaccount.com \
    --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
    --create-disk=auto-delete=yes,boot=yes,device-name=instance-20260914-054744,disk-resource-policy=projects/sesac-dev-400904/regions/us-central1/resourcePolicies/default-schedule-1,image=projects/debian-cloud/global/images/debian-13-trixie-v20260908,mode=rw,size=10,type=pd-balanced \
    --no-shielded-secure-boot \
    --shielded-vtpm \
    --shielded-integrity-monitoring \
    --labels=goog-ops-agent-policy=v2-template-1-7-0,goog-ec-src=vm_add-gcloud \
    --reservation-affinity=any \
&& \
printf "agentsRule:\n  packageState: installed\n  version: latest\ninstanceFilter:\n  inclusionLabels:\n  - labels:\n      goog-ops-agent-policy: v2-template-1-7-0\n" > config.yaml \
&& \
gcloud compute instances ops-agents policies create goog-ops-agent-v2-template-1-7-0-us-central1-c \
    --project=sesac-dev-400904 \
    --zone=us-central1-c \
    --file=config.yaml

Created [https://www.googleapis.com/compute/v1/projects/sesac-dev-400904/zones/us-central1-c/instances/instance-20260914-054744].


NAME                      ZONE           MACHINE_TYPE  PREEMPTIBLE  INTERNAL_IP  EXTERNAL_IP   STATUS
instance-20260914-054744  us-central1-c  e2-medium                  10.128.0.60  34.57.111.41  RUNNING
agentsRule:
  packageState: installed
  version: latest
instanceFilter:
  inclusionLabels:
  - labels:
      goog-ops-agent-policy: v2-template-1-7-0
policyId: projects/902882112756/locations/us-central1-c/osPolicyAssignments/goog-ops-agent-v2-template-1-7-0-us-central1-c
rolloutState: IN_PROGRESS
updateTime: '2026-09-14T06:32:41.840324Z'


## 2. 생성된 인스턴스 확인

In [3]:
!gcloud compute instances list --project=sesac-dev-400904 --filter="name=instance-20260914-054744"

NAME                      ZONE           MACHINE_TYPE  PREEMPTIBLE  INTERNAL_IP  EXTERNAL_IP   STATUS
instance-20260914-054744  us-central1-c  e2-medium                  10.128.0.60  34.57.111.41  RUNNING


## 3. Ops Agent 정책 확인

In [4]:
!gcloud compute instances ops-agents policies describe goog-ops-agent-v2-template-1-7-0-us-central1-c --project=sesac-dev-400904 --zone=us-central1-c

agentsRule:
  packageState: installed
  version: latest
instanceFilter:
  inclusionLabels:
  - labels:
      goog-ops-agent-policy: v2-template-1-7-0
policyId: projects/902882112756/locations/us-central1-c/osPolicyAssignments/goog-ops-agent-v2-template-1-7-0-us-central1-c
rolloutState: IN_PROGRESS
updateTime: '2026-09-14T06:32:41.840324Z'


## 4. 리소스 정리 (과금 방지용 삭제 코드)
실습이 끝난 후 불필요한 과금을 방지하려면 아래 코드의 주석을 해제하고 실행하세요.

In [5]:
# 1. Compute Engine 인스턴스 삭제
!gcloud compute instances delete instance-20260914-054744 --zone=us-central1-c --project=sesac-dev-400904 --quiet

# 2. Ops Agent 정책 삭제 (OS Config 하위 리소스 직접 삭제)
!gcloud compute os-config os-policy-assignments delete goog-ops-agent-v2-template-1-7-0-us-central1-c --location=us-central1-c --project=sesac-dev-400904 --quiet


Deleted [https://www.googleapis.com/compute/v1/projects/sesac-dev-400904/zones/us-central1-c/instances/instance-20260914-054744].
Delete request issued for: [goog-ops-agent-v2-template-1-7-0-us-central1-c]
Waiting for operation [projects/902882112756/locations/us-central1-c/osPolicyAs
signments/goog-ops-agent-v2-template-1-7-0-us-central1-c/operations/c5d840c4-77
f5-471a-a32a-d8a0a9b7fa63] to complete...done.                                 
Deleted OS policy assignment [goog-ops-agent-v2-template-1-7-0-us-central1-c].
